# Unit III — AI in Personalized Medicine & Risk Stratification
**Pharmakaksha · Learn · Grow · Achieve** — BP703T AI in Clinical Applications (B.Pharm Sem VII, PCI NEP 2020)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/BP703T_Unit3_Personalized_Medicine.ipynb)

Every example from the Unit III notes, slides and video, in order, then lab exercises with solutions. Data: `clopidogrel_response.csv`, `diabetes_screening.csv` (fictional teaching data).

## Using learn.pharmakaksha.com
- **First run is slow.** Python starts inside your browser; the first cell can take about 10 seconds. Later cells are fast.
- **Kernel indicator.** The circle at the top right is filled while Python is busy. Wait for it to clear.
- **Stuck?** Use *Kernel → Restart Kernel*, then run the cells again from the top.
- **Saving.** Your changes are saved in this browser only. Use *File → Download* to keep a copy.
- **Start over.** Delete your copy from the file list on the left and reopen the notebook from the course folder.

In [ ]:
import os
import numpy as np
import pandas as pd
BASE = "https://raw.githubusercontent.com/ajreddys/learn-pharmakaksha-site/main/content/BP703T%20AI%20in%20Clinical%20Applications%20%28Theory%29/"

def load(name):
    """Read a practice CSV: local copy first, then the course's GitHub folder."""
    if os.path.exists(name):
        return pd.read_csv(name)
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        from pyodide.http import open_url   # inside JupyterLite
        return pd.read_csv(open_url(BASE + name))

## 3.1–3.2 Precision medicine: covariates and response
600 patients on clopidogrel after a heart attack (fictional). Clopidogrel is a prodrug: CYP2C19 activates it. `responder` = adequate platelet inhibition.

In [ ]:
cl = load("clopidogrel_response.csv")
print("Responders:", round(cl["responder"].mean(), 3))
cl.head()

In [ ]:
cl.groupby("cyp2c19")["responder"].agg(["mean", "size"]).round(2).reindex(["NM", "IM", "PM"])

In [ ]:
import matplotlib.pyplot as plt
rate = cl.groupby("cyp2c19")["responder"].mean().reindex(["NM", "IM", "PM"]) * 100
plt.bar(["Normal", "Intermediate", "Poor"], rate, color=["#1F8A70", "#E07A2E", "#12303A"])
plt.ylabel("Responders (%)"); plt.title("Response by CYP2C19 metaboliser status"); plt.show()

## 3.3–3.4 Logistic regression: responders vs non-responders
One-hot encode genotype and sex, keeping normal metaboliser and male as the reference groups.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

X = pd.get_dummies(cl.drop(columns=["patient_id", "responder"]), columns=["cyp2c19", "sex"], dtype=int)
X = X.drop(columns=["cyp2c19_NM", "sex_M"])
y = cl["responder"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = LogisticRegression(C=1e6, max_iter=5000).fit(X_tr, y_tr)
pd.Series(np.exp(model.coef_[0]), index=X.columns).round(2).rename("odds ratio for responding")

In [ ]:
p_resp = model.predict_proba(X_te)[:, 1]
gene_only = LogisticRegression().fit(X_tr[["cyp2c19_IM", "cyp2c19_PM"]], y_tr)
p_gene = gene_only.predict_proba(X_te[["cyp2c19_IM", "cyp2c19_PM"]])[:, 1]
print("AUC, full model:   ", round(roc_auc_score(y_te, p_resp), 3))
print("AUC, genotype only:", round(roc_auc_score(y_te, p_gene), 3))
for p, lab in [(p_resp, "full"), (p_gene, "genotype only")]:
    f, t, _ = roc_curve(y_te, p); plt.plot(f, t, label=lab)
plt.plot([0, 1], [0, 1], "--", color="grey"); plt.xlabel("1 − specificity"); plt.ylabel("Sensitivity"); plt.legend(); plt.show()

## 3.5 Evaluation metrics: flagging non-responders
The clinical question is *who will not respond?* So the positive class is the non-responder.

In [ ]:
p_non = 1 - p_resp
y_non = 1 - y_te
def evaluate(threshold):
    tn, fp, fn, tp = confusion_matrix(y_non, (p_non >= threshold).astype(int)).ravel()
    return {"threshold": threshold, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "sensitivity": round(tp / (tp + fn), 2), "specificity": round(tn / (tn + fp), 2),
            "precision": round(tp / (tp + fp), 2), "accuracy": round((tp + tn) / len(y_non), 2)}
pd.DataFrame([evaluate(0.5), evaluate(0.3)])

## 3.3 Disease risk: type 2 diabetes screening
2,000 adults (fictional). We compare a logistic model with the Indian Diabetes Risk Score (IDRS), a published points score.

In [ ]:
ds = load("diabetes_screening.csv")
print("Prevalence:", round(ds["diabetes"].mean(), 3))

def idrs(r):
    s = 0 if r.age < 35 else (20 if r.age < 50 else 30)
    lo, hi = (90, 100) if r.sex == "M" else (80, 90)
    s += 0 if r.waist_cm < lo else (10 if r.waist_cm < hi else 20)
    s += {"vigorous": 0, "moderate": 10, "mild": 20, "sedentary": 30}[r.activity]
    s += [0, 10, 20][r.family_history]
    return s

ds["idrs"] = ds.apply(idrs, axis=1)
ds.head()

In [ ]:
Xd = pd.get_dummies(ds[["age", "waist_cm", "family_history", "activity", "urban", "sex"]],
                    columns=["activity", "sex"], dtype=int).drop(columns=["activity_vigorous", "sex_F"])
yd = ds["diabetes"]
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=42, stratify=yd)
dm_model = LogisticRegression(C=1e6, max_iter=5000).fit(Xd_tr, yd_tr)
p_dm = dm_model.predict_proba(Xd_te)[:, 1]
print("AUC, logistic model:", round(roc_auc_score(yd_te, p_dm), 3))
print("AUC, IDRS score:    ", round(roc_auc_score(yd_te, ds.loc[Xd_te.index, "idrs"]), 3))

### Risk stratification and calibration
A good risk model is *calibrated*: of people told 10%, about 10% should have the disease.

In [ ]:
te = ds.loc[Xd_te.index].copy()
te["predicted"] = p_dm
te["band"] = pd.cut(te["predicted"], [0, 0.05, 0.15, 1], labels=["Low <5%", "Intermediate 5–15%", "High >15%"])
te.groupby("band", observed=True).agg(n=("diabetes", "size"), mean_predicted=("predicted", "mean"),
                                       observed_rate=("diabetes", "mean")).round(3)

In [ ]:
# IDRS >= 60 = high risk
pd.crosstab(te["idrs"] >= 60, te["diabetes"], rownames=["IDRS >= 60"], colnames=["diabetes"])

## 3.6 Ethics: does the model work equally well for everyone?

In [ ]:
for col, groups in [("sex", ["M", "F"]), ("urban", [0, 1])]:
    for g in groups:
        i = te[col] == g
        print(f"{col}={g}: n={i.sum()}, AUC={roc_auc_score(te.loc[i, 'diabetes'], te.loc[i, 'predicted']):.3f}")

In [ ]:
# Clopidogrel model by sex: small subgroups give unstable numbers
cl_te = cl.loc[X_te.index]
for s in ["M", "F"]:
    i = (cl_te["sex"] == s).values
    print(s, "n =", i.sum(), " AUC =", round(roc_auc_score(y_te[i], p_resp[i]), 3))

## Worked example: three patients starting clopidogrel

In [ ]:
patients = pd.DataFrame([[62, 25, 1, 1, 0, 0, 1, 0], [62, 25, 1, 1, 0, 0, 0, 0], [62, 25, 0, 0, 0, 0, 0, 0]],
                        columns=X.columns, index=["Mr. A: PM, diabetic, omeprazole", "Mr. B: NM, diabetic, omeprazole", "Mr. C: NM, no diabetes, no PPI"])
patients["P(responds)"] = model.predict_proba(patients[X.columns])[:, 1].round(3)
patients[["P(responds)"]]

---
# Lab exercises
1. Which single covariate (after genotype) lowers the odds of response the most? Explain it pharmacologically.
2. For Mr. B, switch omeprazole off (`ppi = 0`). How much does his predicted response change?
3. Find the non-responder threshold that gives sensitivity ≥ 0.9. What happens to precision?
4. Using IDRS ≥ 60 as a screening test, compute its sensitivity and specificity on the test set.

## Solutions

In [ ]:
# Lab 1
pd.Series(np.exp(model.coef_[0]), index=X.columns).drop(["cyp2c19_IM", "cyp2c19_PM"]).sort_values().round(2)

In [ ]:
# Lab 2
b = patients.loc[["Mr. B: NM, diabetic, omeprazole"], X.columns].copy()
b["ppi"] = 0
print("With omeprazole:", patients.loc["Mr. B: NM, diabetic, omeprazole", "P(responds)"],
      " without:", model.predict_proba(b)[:, 1].round(3)[0])

In [ ]:
# Lab 3
for t in np.arange(0.5, 0.0, -0.01):
    r = evaluate(round(t, 2))
    if r["sensitivity"] >= 0.9:
        print(r); break

In [ ]:
# Lab 4
tab = pd.crosstab(te["idrs"] >= 60, te["diabetes"])
tp, fn = tab.loc[True, 1], tab.loc[False, 1]
tn, fp = tab.loc[False, 0], tab.loc[True, 0]
print("IDRS >= 60  sensitivity:", round(tp / (tp + fn), 2), " specificity:", round(tn / (tn + fp), 2))

**Teaching data only.** Patients are fictional. Real antiplatelet choices follow CPIC/cardiology guidance and the treating team.